# LoRA fine-tuning: Qwen2.5-1.5B-Instruct (Day 3)

**What this does:** teaches the base model to fill in the JSON form from legal excerpts, and to write
"Not specified in the excerpt" instead of guessing. It trains on 80 human-reviewed examples, picks the
checkpoint with the lowest validation loss, then compares base vs. fine-tuned on the 20 validation examples.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `train.jsonl` and `val.jsonl` (folder icon on the left)
3. Runtime → **Run all**
4. At the end, download `lora_results.zip` and unzip it in the project folder

## 1. Check the GPU

In [ ]:
import torch, platform
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
GPU = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f"GPU: {GPU} ({GPU_MEM_GB} GB) · torch {torch.__version__} · Python {platform.python_version()}")

## 2. Install the same library versions as the Mac
Pinned so the adapter trained here loads locally without version errors.
Colab's preinstalled torchao is too old for peft and makes it refuse to load; we don't use it, so it is removed.
The pyarrow / cudf warning is harmless (neither is used here).

In [ ]:
!pip uninstall -y -q torchao
!pip install -q transformers==5.22.1 peft==0.21.2 trl==1.17.1 accelerate==1.15.0 datasets==5.1.0
import transformers, peft, trl
print("transformers", transformers.__version__, "· peft", peft.__version__, "· trl", trl.__version__)

## 3. Settings and data
`QUICK_TEST = True` runs 2 steps on 2 examples, to check everything works in ~2 minutes. Leave it `False` for the real run.
Each example is split into **prompt** (system + user: instructions, question, excerpts) and **completion** (the JSON answer).
The loss is computed on the completion only, so the model learns to *write the answer*, not to memorize the excerpts.

In [ ]:
import json, os, time, random
import numpy as np
from datasets import Dataset

QUICK_TEST = False
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42
OUT_DIR = "lora_results"
os.makedirs(f"{OUT_DIR}/training", exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def load(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    return rows[:2] if QUICK_TEST else rows

train_rows, val_rows = load("train.jsonl"), load("val.jsonl")
to_pc = lambda r: {"prompt": r["messages"][:2], "completion": r["messages"][2:]}
train_ds = Dataset.from_list([to_pc(r) for r in train_rows])
val_ds = Dataset.from_list([to_pc(r) for r in val_rows])
print(f"train {len(train_ds)} · val {len(val_ds)}")
print("\nExample user prompt:\n", train_rows[0]["messages"][1]["content"][:400], "...")
print("\nExample target JSON:\n", train_rows[0]["messages"][2]["content"][:300], "...")

## 4. Load the base model
float16 weights (the T4 has no bfloat16 support). About 3 GB on the GPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to("cuda")
print(f"Loaded {MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e9:.2f}B parameters")

## 5. LoRA settings and trainer
**LoRA** freezes the original model and trains small "adapter" matrices added to each layer (rank 16), about 1% of the weights.
- `target_modules="all-linear"`: adapters on every attention and feed-forward layer
- 3 epochs, learning rate 2e-4, effective batch size 8 (4 × 2 gradient accumulation)
- Validation loss checked every 5 steps; the **best** checkpoint (lowest validation loss) is kept, which guards against overfitting on 80 examples

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
HP = dict(num_train_epochs=3, learning_rate=2e-4, per_device_train_batch_size=4, gradient_accumulation_steps=2,
          lr_scheduler_type="cosine", warmup_steps=3, max_length=1024)

args = SFTConfig(
    output_dir=f"{OUT_DIR}/checkpoints", seed=SEED, fp16=True, report_to="none",
    completion_only_loss=True, logging_steps=1,
    eval_strategy="steps", eval_steps=1 if QUICK_TEST else 5,
    save_strategy="steps", save_steps=1 if QUICK_TEST else 5, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    per_device_eval_batch_size=4, max_steps=2 if QUICK_TEST else -1, **HP,
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                     processing_class=tok, peft_config=LoraConfig(**LORA))
trainer.model.print_trainable_parameters()
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)

## 6. Train
Watch `loss` (training) and `eval_loss` (validation) in the table. Training loss should fall; validation loss should fall and then flatten.

In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
train_out = trainer.train()
TRAIN_MIN = (time.perf_counter() - t0) / 60
PEAK_GB = torch.cuda.max_memory_allocated() / 1e9
print(f"\nTraining time: {TRAIN_MIN:.1f} min · peak GPU memory: {PEAK_GB:.2f} GB")
print("Best checkpoint:", trainer.state.best_model_checkpoint, "· best eval_loss:", trainer.state.best_metric)

## 7. Loss curves and training summary

In [ ]:
import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label="training loss", alpha=0.8)
plt.plot(*zip(*ev), "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("LoRA fine-tuning: Qwen2.5-1.5B-Instruct")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training/loss_curve.png", dpi=150); plt.show()

summary = {
    "model": MODEL_ID, "gpu": GPU, "gpu_memory_gb": GPU_MEM_GB, "quick_test": QUICK_TEST,
    "train_examples": len(train_ds), "val_examples": len(val_ds),
    "lora": LORA, "hyperparameters": HP, "precision": "fp16", "seed": SEED,
    "trainable_params": trainable, "total_params": sum(p.numel() for p in trainer.model.parameters()),
    "steps": trainer.state.global_step, "train_minutes": round(TRAIN_MIN, 1), "peak_gpu_memory_gb": round(PEAK_GB, 2),
    "first_train_loss": tr[0][1], "last_train_loss": tr[-1][1],
    "first_eval_loss": ev[0][1], "best_eval_loss": trainer.state.best_metric,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "peft": peft.__version__, "trl": trl.__version__},
}
json.dump(summary, open(f"{OUT_DIR}/training/training_summary.json", "w"), indent=2)
json.dump(hist, open(f"{OUT_DIR}/training/log_history.json", "w"), indent=2)
print(json.dumps({k: summary[k] for k in ["steps", "train_minutes", "peak_gpu_memory_gb", "first_train_loss",
                                          "last_train_loss", "first_eval_loss", "best_eval_loss"]}, indent=2))

## 8. Before vs. after on the 20 validation examples
Same prompt, greedy decoding (no randomness). The base model = the same network with the adapter switched off.
Checks per answer: valid JSON · exact 9 keys · section matches · `mandatory` matches · quote is word for word in the excerpts ·
says "Not specified" exactly when it should (4 missing-evidence examples).

In [ ]:
import re
KEYS = ["jurisdiction", "topic", "section", "eligibility", "entitlement_or_requirement", "threshold",
        "exceptions", "mandatory", "supporting_quote"]
norm = lambda s: re.sub(r"\s+", " ", str(s)).strip()

def parse(text):
    m = re.search(r"\{.*\}", text, flags=re.S)
    try: return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError: return None

def check(got, exp, prompt_text):
    c = {"valid_json": got is not None}
    if got is None:
        return c | {k: False for k in ["exact_keys", "section_match", "mandatory_match", "quote_ok", "not_specified_ok"]}
    q, eq = got.get("supporting_quote", None), exp["supporting_quote"]
    return c | {
        "exact_keys": set(got) == set(KEYS),
        "section_match": norm(got.get("section")) == exp["section"],
        "mandatory_match": got.get("mandatory") == exp["mandatory"],
        "quote_ok": (q == "" and eq == "") or (bool(q) and norm(q) in norm(prompt_text)),
        "not_specified_ok": exp["section"].startswith("Not specified") == norm(got.get("section", "")).startswith("Not specified"),
    }

def generate(msgs):
    inp = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        out = model_eval.generate(**inp, max_new_tokens=400, do_sample=False)
    return tok.decode(out[0, inp["input_ids"].shape[1]:], skip_special_tokens=True)

model_eval = trainer.model.eval()
rows = []
for r in val_rows:
    msgs, exp = r["messages"][:2], json.loads(r["messages"][2]["content"])
    with model_eval.disable_adapter():
        base_txt = generate(msgs)
    ft_txt = generate(msgs)
    rows.append({"id": r["id"], "type": r["type"],
                 "base": {"output": base_txt, "checks": check(parse(base_txt), exp, msgs[1]["content"])},
                 "fine_tuned": {"output": ft_txt, "checks": check(parse(ft_txt), exp, msgs[1]["content"])},
                 "expected": exp})
    print(r["id"], "done")

metrics = list(rows[0]["base"]["checks"])
table = {m: {"base": sum(x["base"]["checks"][m] for x in rows), "fine_tuned": sum(x["fine_tuned"]["checks"][m] for x in rows)}
         for m in metrics}
missing = [x for x in rows if x["expected"]["section"].startswith("Not specified")]
print(f"\n{'check':18} {'base':>6} {'fine-tuned':>11}   (out of {len(rows)})")
for m, v in table.items():
    print(f"{m:18} {v['base']:>6} {v['fine_tuned']:>11}")
print(f"\nMissing-evidence examples ({len(missing)}): says 'Not specified' → base "
      f"{sum(x['base']['checks']['not_specified_ok'] for x in missing)}, fine-tuned "
      f"{sum(x['fine_tuned']['checks']['not_specified_ok'] for x in missing)}")
json.dump({"summary": table, "n": len(rows), "examples": rows},
          open(f"{OUT_DIR}/training/val_before_after.json", "w"), indent=2, ensure_ascii=False)

### Side by side: V019 (the base model hallucinated here in the local smoke test)

In [ ]:
for x in rows:
    if x["id"] in ("V019", rows[-1]["id"]):
        print(f"=== {x['id']} ===\n--- base ---\n{x['base']['output']}\n--- fine-tuned ---\n{x['fine_tuned']['output']}\n")
        break

## 9. Save the adapter and download everything
The adapter is small (tens of MB). The base model is not included; it is re-downloaded from Hugging Face when needed.

In [ ]:
import shutil
trainer.save_model(f"{OUT_DIR}/lora-adapter")          # saves the best checkpoint's adapter
shutil.rmtree(f"{OUT_DIR}/checkpoints", ignore_errors=True)
shutil.make_archive("lora_results", "zip", OUT_DIR)
print("Contents:", sorted(os.listdir(OUT_DIR)), "/", sorted(os.listdir(f"{OUT_DIR}/lora-adapter")))
print(f"lora_results.zip: {os.path.getsize('lora_results.zip')/1e6:.1f} MB")
try:
    from google.colab import files
    files.download("lora_results.zip")
except ImportError:
    pass